In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc 
import anndata as ad
import h5py 
import glob
from downstream import *
import matplotlib.pyplot as plt
import glob
import os

In [ ]:
sc.settings.verbosity = 3             
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')
sc.set_figure_params(scanpy=True, figsize=(8,8))     

In [ ]:
# Set base path for accessing files 
base_path = '/home/EOCRC_atlas/'

In [ ]:
# Set up to collect QC metrics per sample 
qc_metrics = pd.DataFrame(columns = ['cells', 'median_genes', 'median_counts', 'median_pct_mito'])

In [ ]:
# Save path locations for cellbender counts and cumulus adata objects 
h5ad_files = sorted(glob.glob(os.path.join(base_path,'data/cumulus/*.h5ad')))  #cellbender cumulus data in anndata object
h5_files = sorted(glob.glob(os.path.join(base_path,'data/cellbender/*.h5')))    #cellbender raw data

In [ ]:
# Load in each individual sample, save as an adata object, and store QCs for that sample 
for filename in h5_files:   
    adata = anndata_from_h5(filename)
    sample = filename
    sample = sample.replace(os.path.join(base_path, 'data/cellbender/'), '').replace('_out_FPR_0.01_filtered.h5', '')
    print(sample) 
    print(type(sample))
    adata.obs.index = adata.obs.index.map(lambda x: '{}-'.format(sample) + x[:-2])
    adata.var_names_make_unique()
    adata.var['mt'] = adata.var_names.str.startswith('MT-') 
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    sc.pp.filter_cells(adata, min_counts=400)
    sc.pp.filter_cells(adata, min_genes=200)
    adata = adata[adata.obs.pct_counts_mt < 50, :]
    qc_metrics.loc[len(qc_metrics)] = [adata.n_obs, adata.obs['n_genes'].median(0), adata.obs['n_counts'].median(0), adata.obs['pct_counts_mt'].median(0)]
    adata.write_h5ad(os.path.join(base_path, 'data/cellbender_h5ad', f'cellbender_{sample}.h5ad'))

In [ ]:
# add QCsto master sheet
qc_metrics.index = h5_files
qc_metrics.index = qc_metrics.index.map(lambda n: n.replace(os.path.join(base_path, 'data/cellbender/'), '').replace('_out_FPR_0.01_filtered.h5', ''))
qc_metrics.to_csv(os.path.join(base_path, 'results/filtered_postcellbender_QCs.csv'))

In [ ]:
# Merge filtered cellbender adata objects into a single adata object containing all samples 
# Check the size of the final adata object 
# Save final merged adata object 
files = glob.glob(os.path.join(base_path, 'data/cellbender_h5ad/*.h5ad'))
adatas = []
for filename in files: 
    tmp = sc.read_h5ad(filename)
    adatas.append(tmp)

adata_all = ad.concat(adatas, join="outer")
adata_all.obs_names_make_unique()
print(adata_all.shape)
adata_all.write_h5ad(os.path.join(base_path, 'data/all_samples.h5ad'))

In [ ]:
# Optional jump in point if analysis is restarted
adata_all = sc.read_h5ad(os.path.join(base_path, 'data/all_samples.h5ad'))

In [ ]:
# Collect metadata from cumulus h5ad objects to pull doublet information
# Save metadata data for each individual cumulus object as its own .csv file 
for filename in h5ad_files: 
    adata = sc.read_h5ad(filename)
    sample = filename
    sample = sample.replace(os.path.join(base_path, 'data/cumulus/'), '').replace('.GRCh38-rna.h5ad', '')
    #print(sample)
    metadata_cumulus = adata.obs
    metadata_cumulus.to_csv(os.path.join(base_path, 'results/cumulus_metadata/metadata_cumulus_{}.csv'.format(sample)))

In [ ]:
# Combine the cumulus metadata .csv files into one file
meta_files = glob.glob(os.path.join(base_path, 'results/cumulus_metadata/*T[0-9].csv'))
all_meta = pd.concat([pd.read_csv(f) for f in meta_files])
all_meta.to_csv(os.path.join(base_path, 'results/cumulus_metadata_all.csv'), index=False)
all_meta.set_index('barcodekey', inplace=True)
all_meta = all_meta.reindex(adata_all.obs.index)
all_meta

In [ ]:
# Check that metrics that exist in cumulus and cellbender objects match 
# The printed number should match the total number of cells in the merged adata object 
print(sum(adata_all.obs.index == all_meta.index))
print(sum(adata_all.obs['n_genes']==all_meta['n_genes']))
print(sum(adata_all.obs['n_counts']==all_meta['n_counts']))

In [ ]:
# Add doublet and Sample ID info to adata.obs (pulled from cumulus metadata)
# Check adata.obs
adata_all.obs['pred_dbl'] = all_meta['pred_dbl']
adata_all.obs['doublet_score'] = all_meta['doublet_score']
adata_all.obs['Channel'] = all_meta['Channel'] # Should probably rename channel to be FRID
adata_all.obs

In [ ]:
# Remove doublets 
adata_all = adata_all[adata_all.obs.pred_dbl == False, :]
print(adata_all.shape)

In [ ]:
# Save filtered atlas prior to processing 
adata_all.write_h5ad(os.path.join(base_path, 'data/all_samples_noDoublets.h5ad'))

In [ ]:
# Optional jump in point if analysis is restarted
adata_all = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_noDoublets.h5ad'))

In [ ]:
# Rename one patient with SampleID mistake - make sure to update raw files too 
# COLFR6330_T1 --> COLFR6339_T1 
adata_all.obs.index = adata_all.obs.index.str.replace('COLFR6330_T1', 'COLFR6339_T1')

for col in adata_all.obs.columns:
    if isinstance(adata_all.obs[col].dtype, pd.CategoricalDtype):
        if 'COLFR6330_T1' in adata_all.obs[col].cat.categories:
            adata_all.obs[col] = adata_all.obs[col].cat.rename_categories(lambda x: 'COLFR6339_T1' if x == 'COLFR6330_T1' else x)
    else:
        adata_all.obs[col] = adata_all.obs[col].replace('COLFR6330_T1', 'COLFR6339_T1')

In [ ]:
# Filter out high UMI and high gene cells that remain after doublet filtering 
# on log axis
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 3))
ax1.hist(adata_all.obs['total_counts'], bins=500, log=True)
ax2.hist(adata_all.obs['n_genes_by_counts'], bins=1000, log=True)
ax3.hist(adata_all.obs['pct_counts_mt'], bins=1000, log=True)

# not log axis 
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 3))
ax1.hist(adata_all.obs['total_counts'], bins=500, log=False)
ax2.hist(adata_all.obs['n_genes_by_counts'], bins=1000, log=False)
ax3.hist(adata_all.obs['pct_counts_mt'], bins=1000, log=False)

In [ ]:
# calculate 95th percentile for these QCs
print("99.9th percentile for nUMI:", np.percentile(adata_all.obs['total_counts'], 99.5))
print("99.9th percentile for nGene:", np.percentile(adata_all.obs['n_genes_by_counts'], 99.5))
print("99.9th percentile for % mito:", np.percentile(adata_all.obs['pct_counts_mt'], 99.5))

In [ ]:
adata = adata_all.copy()
del adata_all

In [ ]:
# remove cells with too many genes or UMI counts 
tmp1 = adata.shape
adata = adata[adata.obs["n_genes"] < 10000, :]
adata = adata[adata.obs["n_counts"] < 50000, :]
tmp2 = adata.shape 
removed = tmp1[0]-tmp2[0]
print(removed, ' cells were removed') # 2,134 nuclei filtered out in total

In [ ]:
# Normalize the data and store normalized data as its own layer for each normalization step 
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, inplace = True, target_sum=1e4)
adata.layers['norm_counts'] = adata.X.copy()
sc.pp.log1p(adata)
adata.layers['log_counts'] = adata.X.copy()

In [ ]:
# Check that values are as expected 
print(adata.layers['counts'][0:20,0:20])
print(adata.layers['norm_counts'][0:20,0:20])
print(adata.layers['log_counts'][0:20,0:20])

In [ ]:
# Calculate and plot highly variable genes 
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata)

In [ ]:
# Store a copy of adata in the .raw field before subsetting to just variable genes (NOT RAW COUNTS)
adata.raw = adata

In [ ]:
# Subset to just variable genes 
# Scale the data 
adata = adata[:, adata.var.highly_variable]
sc.pp.scale(adata, max_value=10)  

In [ ]:
# Run PCA and generate PCA plots 
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca(adata, color='n_genes')
sc.pl.pca_variance_ratio(adata, log=True)

In [ ]:
# Calculate nearest neighbors 
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

In [ ]:
# Calculate UMAP 
sc.tl.umap(adata)

In [ ]:
# Calculate and plot leiden clusters 
sc.tl.leiden(adata, resolution=.1, key_added = 'leiden_res.1')

In [ ]:
sc.pl.umap(adata, color=['leiden_res.1'], size=1)

In [ ]:
# Save object 
adata.write_h5ad(os.path.join(base_path, 'data/all_samples_processed.h5ad'))

In [ ]:
# Plot QCs (data has been thresholded for better visibility on UMAP - thresholds can be adjusted)

# Plot and save nGenes on UMAP 
fig, ax = plt.subplots()
sc.pl.umap(adata, color=['n_genes'], use_raw=False, size=2, cmap='inferno', vmin='p15', ax=ax)
fig.savefig(os.path.join(base_path, 'results/QCs/nGenes_UMAP.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

# Plot and save nCounts on UMAP
fig, ax = plt.subplots()
sc.pl.umap(adata, color=['n_counts'], use_raw=False, size=2, cmap='inferno', vmin='p15', ax=ax)
fig.savefig(os.path.join(base_path, 'results/QCs/nCounts_UMAP.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

# Plot and save % mito on UMAP 
fig, ax = plt.subplots()
sc.pl.umap(adata, color=['pct_counts_mt'], use_raw=False, size=2, cmap='inferno', vmin='p15', ax=ax)
fig.savefig(os.path.join(base_path, 'results/QCs/pctMito_UMAP.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
# Optional jump in point if analysis is restarted
adata = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_processed.h5ad'))

In [ ]:
# Add patient metadata 
# Rename Channel to FRID 
adata.obs = adata.obs.rename(columns={'Channel': 'FRID'})

df = pd.read_csv(os.path.join(base_path, 'docs/metadata.csv'),
                keep_default_na=False)

# First, convert the 'FRID' column in 'adata.obs' to string if it's not already,
# to ensure matching types with the DataFrame
adata.obs['FRID'] = adata.obs['FRID'].astype(str)

# Create a dictionary from the DataFrame for quick lookup
age_dict = pd.Series(df['Age'].values,index=df['FR ID']).to_dict()
race_dict = pd.Series(df['Race'].values,index=df['FR ID']).to_dict()
sex_dict = pd.Series(df['Pt Sex'].values,index=df['FR ID']).to_dict()
decade_dict = pd.Series(df['Decade of surgery'].values,index=df['FR ID']).to_dict()
ethnicity_dict = pd.Series(df['Ethnicity'].values,index=df['FR ID']).to_dict()
site_dict = pd.Series(df['Cancer Primary Site ICD-O-3'].values,index=df['FR ID']).to_dict()
stage_dict = pd.Series(df['Overall Group Stage at Initial Diagnosis'].values,index=df['FR ID']).to_dict()
therapy_dict = pd.Series(df['Did the patient receive neoadjuvant chemotherapy or radiation therapy before pathological stage diagnosis?'].values,index=df['FR ID']).to_dict()
pathT_dict = pd.Series(df['Path T Stage'].values,index=df['FR ID']).to_dict()
pathN_dict = pd.Series(df['Path N Stage'].values,index=df['FR ID']).to_dict()
msi_dict = pd.Series(df['Select the stated MSI result'].values,index=df['FR ID']).to_dict()
lynch_dict = pd.Series(df['Lynch'].values,index=df['FR ID']).to_dict()

# Map the 'Age' from the DataFrame to 'adata.obs' using the 'FRID' field
adata.obs['Age'] = adata.obs['FRID'].map(age_dict)
adata.obs['Race'] = adata.obs['FRID'].map(race_dict)
adata.obs['Sex'] = adata.obs['FRID'].map(sex_dict)
adata.obs['Decade'] = adata.obs['FRID'].map(decade_dict)
adata.obs['Ethnicity'] = adata.obs['FRID'].map(ethnicity_dict)
adata.obs['Site'] = adata.obs['FRID'].map(site_dict)
adata.obs['Stage'] = adata.obs['FRID'].map(stage_dict)
adata.obs['Therapy'] = adata.obs['FRID'].map(therapy_dict)
adata.obs['PathT'] = adata.obs['FRID'].map(pathT_dict)
adata.obs['PathN'] = adata.obs['FRID'].map(pathN_dict)
adata.obs['MSI'] = adata.obs['FRID'].map(msi_dict)
adata.obs['Lynch'] = adata.obs['FRID'].map(lynch_dict)

# Rename colon areas to L and R 
adata.obs['Site'] = adata.obs['Site'].str.replace('\xa0\xa0\xa0', '') # fixing some weird formatting that happened 
adata.obs['Site'] = adata.obs['Site'].str.replace('    ', ' ')

adata.obs['Sidedness'] = "MISSING"
adata.obs['Sidedness'][adata.obs['Site']=="C18.0 Cecum"] = "Right"
adata.obs['Sidedness'][adata.obs['Site']=="C18.2 Ascending colon"] = "Right"
adata.obs['Sidedness'][adata.obs['Site']=="C18.3 Hepatic flexure of colon"] = "Right"
adata.obs['Sidedness'][adata.obs['Site']=="C18.4 Transverse colon"] = "Right"
adata.obs['Sidedness'][adata.obs['Site']=="C18.5 Splenic flexure of colon"] = "Right"
adata.obs['Sidedness'][adata.obs['Site']=="C18.6 Descending colon"] = "Left"
adata.obs['Sidedness'][adata.obs['Site']=="C18.7 Sigmoid colon"] = "Left"
adata.obs['Sidedness'][adata.obs['Site']=="C19.9 Rectosigmoid junction"] = "Rectal"
adata.obs['Sidedness'][adata.obs['Site']=="C20.9 Rectum NOS"] = "Rectal"
adata.obs['Sidedness'][adata.obs['Site']=="C18.9 Colon NOS"] = "Colon NOS"

# Regroup MSI into MSI-H and MSS 
adata.obs['MSI_v2'] = "MISSING"
adata.obs['MSI_v2'][adata.obs['MSI']=="MSI - Inconclusive"] = "Unknown"
adata.obs['MSI_v2'][adata.obs['MSI']=="Unknown"] = "Unknown"
adata.obs['MSI_v2'][adata.obs['MSI']=="MSI-L: LOW or instability in < 30% of microsatellite markers"] = "MSS: STABLE"
adata.obs['MSI_v2'][adata.obs['MSI']=="Not completed"] = "Unknown"
adata.obs['MSI_v2'][adata.obs['MSI']=="MSI-H: HIGH"] = "MSI-H: HIGH"
adata.obs['MSI_v2'][adata.obs['MSI']=="MSS: STABLE"] = "MSS: STABLE"
adata.obs['MSI_v2'][adata.obs['MSI']==""] = "Unknown"

# Rename treatment status variables 
adata.obs['Therapy_v2'] = "MISSING"
adata.obs['Therapy_v2'][adata.obs['Therapy']=="Not Applicable"] = "No"
adata.obs['Therapy_v2'][adata.obs['Therapy']=="Not Applicable "] = "No"
adata.obs['Therapy_v2'][adata.obs['Therapy']=="No"] = "No"
adata.obs['Therapy_v2'][adata.obs['Therapy']=="Yes"] = "Yes"

# Simplify T stage 
adata.obs['PathT_v2'] = "MISSING"
adata.obs['PathT_v2'][adata.obs['PathT']=="Tis"] = "Tis"
adata.obs['PathT_v2'][adata.obs['PathT']=="T1"] = "T1"
adata.obs['PathT_v2'][adata.obs['PathT']=="T2"] = "T2"
adata.obs['PathT_v2'][adata.obs['PathT']=="pT2"] = "T2"
adata.obs['PathT_v2'][adata.obs['PathT']=="T3"] = "T3"
adata.obs['PathT_v2'][adata.obs['PathT']=="T4a"] = "T4"
adata.obs['PathT_v2'][adata.obs['PathT']=="T4b"] = "T4"
adata.obs['PathT_v2'][adata.obs['PathT']=="Unknown"] = "Unknown"

# Simplify N stage 
adata.obs['PathN_v2'] = "MISSING"
adata.obs['PathN_v2'][adata.obs['PathN']=="N0"] = "N0"
adata.obs['PathN_v2'][adata.obs['PathN']=="N1a"] = "N1"
adata.obs['PathN_v2'][adata.obs['PathN']=="N1b"] = "N1"
adata.obs['PathN_v2'][adata.obs['PathN']=="N1c"] = "N1"
adata.obs['PathN_v2'][adata.obs['PathN']=="N1"] = "N1"
adata.obs['PathN_v2'][adata.obs['PathN']=="N2a"] = "N2"
adata.obs['PathN_v2'][adata.obs['PathN']=="N2b"] = "N2"
adata.obs['PathN_v2'][adata.obs['PathN']=="pN2b"] = "N2"
adata.obs['PathN_v2'][adata.obs['PathN']=="N2"] = "N2"
adata.obs['PathN_v2'][adata.obs['PathN']=="Not Applicable"] = "Unknown"
adata.obs['PathN_v2'][adata.obs['PathN']=="Unknown"] = "Unknown"

# Assign age cohort information 
adata.obs['Cohort'] = False
adata.obs['Cohort'][adata.obs['Decade']=="20-29"]= True
adata.obs['Cohort'][adata.obs['Decade']=="30-39"] = True
adata.obs['Cohort'][adata.obs['Decade']=="40-49"] = True
adata.obs['Cohort'][adata.obs['Cohort']==True] = "UnderFifty"
adata.obs['Cohort'][adata.obs['Cohort']==False] = "FiftyPlus"

In [ ]:
# Check clinical variables 
print(adata.obs['Sidedness'].value_counts())
print(adata.obs['MSI_v2'].value_counts())
print(adata.obs['Therapy_v2'].value_counts())
print(adata.obs['Decade'].value_counts())
print(adata.obs['Cohort'].value_counts())
print(adata.obs['PathT_v2'].value_counts())
print(adata.obs['PathN_v2'].value_counts())
print(adata.obs['Sex'].value_counts())

In [ ]:
# save processed object with metadata 
adata.write_h5ad(os.path.join(base_path, 'data/all_samples_processed.h5ad'))

In [ ]:
# print QC stats 
print("check filters")
print(f"min nGene : {adata.obs.n_genes_by_counts.min()}\nmin nUMI : {adata.obs.total_counts.min()}\n" \
      f"max pMT {adata.obs.pct_counts_mt.max()}\nmax nUMI : {adata.obs.total_counts.max()}\n" \
      f"max nGene {adata.obs.n_genes_by_counts.max()}")
print("\nmedian values")
print(f"median nGene : {adata.obs.n_genes_by_counts.median()}\nmedian nUMI : {adata.obs.total_counts.median()}\n" \
      f"median pMT {adata.obs.pct_counts_mt.median()}")

In [ ]:
# optional jump in point
adata = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_processed.h5ad'))

In [ ]:
# Create simplifed sample metadata table 
metadata = adata.obs[['FRID', 'Decade', 'Cohort', 'Sidedness', 'Sex', 'PathT_v2', 'PathN_v2', 'MSI_v2', 'Therapy_v2', 'Age', 'Survival', 'Recurrance']]
metadata.index = metadata['FRID']
metadata = metadata.drop_duplicates(keep='first')
metadata.to_csv(os.path.join(base_path, 'results/simple_metadata_07-08-2025.csv'))
metadata

In [ ]:
# load raw version of the atlas 
adata_raw = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_noDoublets.h5ad'))

In [ ]:
# Rename one patient with SampleID mistake - make sure to update raw files too 
# COLFR6330_T1 --> COLFR6339_T1 
adata_raw.obs.index = adata_raw.obs.index.str.replace('COLFR6330_T1', 'COLFR6339_T1')

for col in adata_raw.obs.columns:
    if isinstance(adata_raw.obs[col].dtype, pd.CategoricalDtype):
        if 'COLFR6330_T1' in adata_raw.obs[col].cat.categories:
            adata_raw.obs[col] = adata_raw.obs[col].cat.rename_categories(lambda x: 'COLFR6339_T1' if x == 'COLFR6330_T1' else x)
    else:
        adata_raw.obs[col] = adata_raw.obs[col].replace('COLFR6330_T1', 'COLFR6339_T1')

In [ ]:
# remove cells FROM RAW DATA with too many genes or UMI counts 
tmp1 = adata_raw.shape
adata_raw = adata_raw[adata_raw.obs["n_genes"] < 10000, :]
adata_raw = adata_raw[adata_raw.obs["n_counts"] < 50000, :]
tmp2 = adata_raw.shape 
removed = tmp1[0]-tmp2[0]
print(removed, ' cells were removed') # 2,134 nuclei filtered out in total

In [ ]:
# check that sizes check out 
print(sum(adata_raw.obs_names==adata.obs_names))
print(adata_raw.shape)
print(adata.shape)

In [ ]:
# transfer obsm and obs to the raw data object
adata_raw.obs = adata.obs 
adata_raw.obsm = adata.obsm

In [ ]:
# save raw object with metadata 
adata_raw.write_h5ad(os.path.join(base_path, 'data/all_samples_raw_withProcessedInfo.h5ad'))